In [2]:
import sys
import numpy as np
import pandas as pd
import gamspy as gp

from src.parameters import *

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container(options=gp.Options(relative_optimality_gap=0))

In [3]:
# LOAD DATA
teams_df = pd.read_csv("data/processed/hq.csv")
circuits_df = pd.read_csv("data/processed/circuits.csv")
distances_df = pd.read_csv("data/processed/distances.csv")
climate_df = pd.read_csv("data/processed/climate.csv")
# festivals_df = pd.read_csv("data/processed/festivals.csv")

In [4]:
# CONSTANTS
BREAK_START_DATE = 24 # 2026-08-06 (in climate_df)
BREAK_END_DATE = 26   # 2026-08-30 (in climate_df) 
BREAK_POINT = 14      # NO. OF RACES BEFROE BREAK

In [5]:
# PREPROCESSING

teams = ['ALL'] # teams_df["team_id"].tolist()
circuits = circuits_df["circuit_id"].tolist()

weekends = climate_df['week_num'].astype(str).tolist()
summer_break = climate_df.loc[
    climate_df['week_num'].between(BREAK_START_DATE, BREAK_END_DATE), 
    'week_num'].astype(str).tolist()

race_emissions = distances_df[distances_df['from'].isin(circuits)][["from", "to", "emissions_kgCO2e"]]
hq_emissions = distances_df[distances_df['from'].isin(teams)][["from", "to", "emissions_kgCO2e"]]

In [6]:
n_teams = 1 # 10
n_races = len(circuits)
n_weekends = len(weekends)

In [7]:
feasible_dates = np.zeros((n_weekends, n_races))
feasible_dates_df = pd.DataFrame(feasible_dates, columns=circuits, index=climate_df['week_num'].astype(str).tolist())

climate_df.index = climate_df['week_num'].astype(str).tolist()

for circuit in circuits:
    temp_col = f"{circuit}_avg_temp"
    precip_col = f"{circuit}_avg_precip"

    ok_temp = climate_df[temp_col].between(MIN_TEMP_F, MAX_TEMP_F)
    ok_precip = climate_df[precip_col] < MAX_PRECIP_IN

    feasible_dates_df.loc[ok_temp & ok_precip, circuit] = 1

In [8]:
# SET
Weekend = gp.Set(m, records=weekends)
SummerBreak = gp.Set(m, domain=[Weekend], records=summer_break)
Team = gp.Set(m, records=teams)
Circuit = gp.Set(m, records=circuits)
i = gp.Alias(m, alias_with=Circuit)
j = gp.Alias(m, alias_with=Circuit)
t = gp.Alias(m, alias_with=Weekend)



# PARAMETERS
r_emissions = gp.Parameter(m, domain=[Circuit, Circuit], records=race_emissions,
                         description="emissions between races in kgCO2e")
hq_emissions = gp.Parameter(m, domain=[Team, Circuit], records=hq_emissions,
                         description="emissions from HQ to race in kgCO2e")

feasible_dates = gp.Parameter(m, domain=[Weekend, Circuit], records=feasible_dates_df.stack().reset_index().values.tolist(),
                            description="feasibility of holding race on weekend (1 if feasible, 0 otherwise)")
feasible_dates[SummerBreak, Circuit] = 0 # No races during the break


first_set = gp.Parameter(m, records=14,
                         description='Number of races before the summer break')
break_start_date = gp.Parameter(m, records=np.array(BREAK_START_DATE),
                                description='Weekend where break begins')
break_end_date = gp.Parameter(m, records=np.array(BREAK_END_DATE),
                                description='Weekend where break ends')


In [13]:
SummerBreak.records

,Weekend,element_text
0,24,
1,25,
2,26,


In [9]:
# VARIABLES
x = gp.Variable(m,type='binary',domain=[i,j],
                description="1 if race in i is scheduled immediately before j, 0 otherwise")
y = gp.Variable(m, type='binary', domain=[Circuit,Weekend], 
                description='1 if race in Circuit is scheduled on Weekend, 0 otherwise')
u = gp.Variable(m,type='positive',domain=[Circuit], 
                description="Position of race in the calendar sequence")

x.fx[i,j].where[i == j] = 0

u.lo[Circuit] = 2                   # all races but first must be at least position 2
u.up[Circuit] = gp.Card(Circuit)    # all races must be at most position number of races

u.fx[Circuit].where[Circuit == "AUS"] = 1                # First race = AUS
u.fx[Circuit].where[Circuit == "ABD"] = gp.Card(Circuit) # Last race = ABD (position = number of races)



In [10]:
# EQUATIONS
# Obvious ones first
assign1 = gp.Equation(m, domain=[j],
                      description='Each circuit can be reached only from one other circuit')
assign1[j]= gp.Sum(i, x[i,j]) == 1

assign2 = gp.Equation(m, domain=[i],
                      description='Can only go to one other Circuit from each circuit')
assign2[i]= gp.Sum(j, x[i,j]) == 1

assign3 = gp.Equation(m, domain=[i],
                      description='Each race is only conducted once')
assign3[i]= gp.Sum(t, y[i,t]) == 1

assign4 = gp.Equation(m, domain=[t],
                      description='A weekend can conduct atmost 1 race')
assign4[t]= gp.Sum(i, y[i,t]) <= 1

mtz = gp.Equation(m,domain=[i,j],
                  description='This is the MTZ equation')
mtz[i,j].where[(i.ord > 1) & (j.ord > 1)]= (
  u[i] - u[j] + 1 <= (gp.Card(i) - 1) * (1 - x[i,j]))

start_cons = gp.Equation(m)
start_cons[...] = gp.Sum(i, x[i, "AUS"]) == 0

end_cons = gp.Equation(m)
end_cons[...] = gp.Sum(j, x["ABD", j]) == 0


In [11]:
# Time Constraints
time_cons1 = gp.Equation(m, domain=[i,t],
                         description='Race can be held only if the weekend is feasible')
time_cons1[i,t] = y[i,t] <= feasible_dates[t,i]

time_cons2 = gp.Equation(m, domain=[i, j],
    description="If j follows i, then weekend(j) must be after weekend(i)")
time_cons2[i, j] = (
    gp.Sum(t, t.ord * y[j, t]) -
    gp.Sum(t, t.ord * y[i, t])
    >= 1 - (gp.Card(i) - 1) * (1 - x[i, j])
)


time_cons3 = gp.Equation(m,
                         description='Only K races until break')
time_cons3[...] = gp.Sum([i, t], y[i, t]*(t.ord < BREAK_START_DATE)) == first_set
